# Домашнее задание: вакансии, отклики и приглашения

Подготовьте два небольших отчёта: **по городам вакансий** и **по размеру работодателя**. Нужно сравнить число вакансий, откликов и приглашений, долю приглашений среди откликов и медиану нижней границы зарплаты в рублях, количество и долю вакансий с непустой рублёвой зарплатой.

Вы сами соберёте таблицы из нескольких источников, посчитаете показатели через `groupby`, построите график и оформите результат. Все необходимые приёмы есть в лекции; SQL-запросы писать не нужно.

**Основная часть — все 8 пунктов.** Ориентир: 1,5–2,5 часа, возможно в несколько подходов. Это предварительная оценка, ещё не проверенная на студентах. В конце есть необязательные упражнения.

Откройте тетрадку из папки, в которой она сохранена. Рядом должны быть папки `data` и `outputs`. Выполняйте ячейки сверху вниз. Подсказки указывают методы и подход; готовые выражения можно посмотреть в лекции.

Сдайте эту тетрадку с кодом, сохранёнными результатами и короткими ответами, а также Excel-файл отчёта. Достаточно обычного последовательного кода: свои функции и циклы не требуются.

## Какие данные есть

Все CSV находятся в `data/sql/`.

| Файл | Что означает одна строка | Важные столбцы |
|---|---|---|
| `vacancies.csv` | Вакансия | `vacancy_id`, `employer_id`, `area_id`, `salary_from`, `currency`, `published_at` |
| `responses.csv` | Отклик | `response_id`, `vacancy_id`, `status` |
| `employers.csv` | Работодатель | `employer_id`, `employer_name`, `area_id` |
| `areas.csv` | Город | `area_id`, `area_name` |
| `employer_size_groups.csv` | Работодатель с известной группой размера | `employer_id`, `size_group` |

`status == "invited"` означает приглашение. Все статусы откликов учитываются в общем числе откликов.

`size_group`: `small` — менее 250 сотрудников, `medium` — 250–999, `large` — от 1000. Это учебные группы, уже рассчитанные в справочнике. Для одной компании группа неизвестна.

В `vacancies.area_id` указан **город вакансии**, а в `employers.area_id` — город работодателя. Они могут различаться. В отчёте нужен город вакансии.

`salary_from` — заявленная нижняя граница зарплаты. Пропуск означает, что сумма неизвестна; это не нулевая зарплата.

In [ ]:
import pandas as pd
import numpy as np

## 1. Загрузите данные и осмотрите основные таблицы

1. Прочитайте пять CSV в переменные `vacancies`, `responses`, `employers`, `areas`, `size_groups`.
2. Для **`vacancies` и `responses`** выведите первые строки, размеры и общую информацию о столбцах.
3. Посчитайте пропуски в `vacancies["salary_from"]` и количество откликов каждого `status`.
4. Небольшие справочники `areas` и `size_groups` выведите целиком.

**Что получить:** представление о структуре данных, число вакансий и откликов, число неизвестных зарплат и распределение статусов. Проверять все ключи и все категории каждой таблицы не нужно.

**Методы:** `pd.read_csv`, `head`, `shape`, `info`, `isna`, `sum`, `value_counts`.

In [ ]:
# 1. Чтение и осмотр
# решение

vacancies = pd.read_csv("vacancies.csv")
responses = pd.read_csv("responses.csv")
employers = pd.read_csv("employers.csv")
areas = pd.read_csv("areas.csv")
size_groups = pd.read_csv("employer_size_groups.csv")

print("вакансии")
display(vacancies.head())
print(f"Размер таблицы (shape): {vacancies.shape}")
print("\nОбщая инфа:")
vacancies.info()

print("\n" + "-"*50 + "\n")

print("отклики")
display(responses.head())
print(f"Размер таблицы (shape): {responses.shape}")
print("\nОбщая инфа:")
responses.info()

missing_salaries = vacancies["salary_from"].isna().sum()
print(f"\nКоличество пропусков в 'salary_from': {missing_salaries}")

status_counts = responses["status"].value_counts()
print("\nКоличество откликов по статусам:")
display(status_counts)


print("\nсправочник городов")
display(areas)

print("\nсправочник размеров")
display(size_groups)

вакансии


,vacancy_id,employer_id,area_id,vacancy_name,professional_role,experience,employment,salary_from,salary_to,currency,published_at,closed_at
0,1001,101,1,Стажёр: аналитик данных,Аналитик данных,Нет опыта,Стажировка,35000.0,45000.0,RUR,2026-05-14 06:00:00,NaN
1,1002,101,1,Senior аналитик данных,Аналитик данных,От 3 до 6 лет,Полная занятость,140000.0,165000.0,RUR,2026-05-29 06:00:00,NaN
2,1003,101,1,Senior аналитик данных,Аналитик данных,От 3 до 6 лет,Полная занятость,135000.0,165000.0,RUR,2026-06-23 06:00:00,2026-07-08 23:59:00
3,1004,101,1,Стажёр: аналитик данных,Аналитик данных,Нет опыта,Стажировка,40000.0,50000.0,RUR,2026-01-13 06:00:00,NaN
4,1005,102,1,Стажёр: аналитик данных,Аналитик данных,Нет опыта,Стажировка,40000.0,55000.0,RUR,2026-06-03 06:00:00,NaN


Размер таблицы (shape): (52, 12)

Общая инфа:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   vacancy_id         52 non-null     int64  
 1   employer_id        52 non-null     int64  
 2   area_id            52 non-null     int64  
 3   vacancy_name       52 non-null     object 
 4   professional_role  52 non-null     object 
 5   experience         52 non-null     object 
 6   employment         52 non-null     object 
 7   salary_from        44 non-null     float64
 8   salary_to          38 non-null     float64
 9   currency           44 non-null     object 
 10  published_at       52 non-null     object 
 11  closed_at          19 non-null     object 
dtypes: float64(2), int64(3), object(7)
memory usage: 5.0+ KB

--------------------------------------------------

отклики


,response_id,vacancy_id,resume_id,created_at,status
0,50001,1001,900773,2026-07-07 08:30:00,rejected
1,50002,1001,901030,2026-05-31 08:40:00,rejected
2,50003,1001,901138,2026-05-23 08:50:00,rejected
3,50004,1001,900709,2026-06-30 07:00:00,viewed
4,50005,1001,900242,2026-06-15 07:10:00,invited


Размер таблицы (shape): (628, 5)

Общая инфа:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 628 entries, 0 to 627
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   response_id  628 non-null    int64 
 1   vacancy_id   628 non-null    int64 
 2   resume_id    628 non-null    int64 
 3   created_at   628 non-null    object
 4   status       628 non-null    object
dtypes: int64(3), object(2)
memory usage: 24.7+ KB

Количество пропусков в 'salary_from': 8

Количество откликов по статусам:


,count
status,
rejected,260
viewed,182
new,94
invited,92



справочник городов


,area_id,area_name,region
0,1,Москва,Центральный ФО
1,2,Санкт-Петербург,Северо-Западный ФО
2,3,Екатеринбург,Уральский ФО
3,4,Новосибирск,Сибирский ФО
4,5,Казань,Приволжский ФО



справочник размеров


,employer_id,size_group
0,101,large
1,102,large
2,103,small
3,104,medium
4,105,large
5,106,medium
6,107,medium
7,108,large
8,109,small
9,110,small


## 2. Подготовьте зарплаты и дату публикации

1. Преобразуйте `vacancies["published_at"]` в дату/время.
2. Создайте в `vacancies` столбец `salary_rur`: значение из `salary_from`, если `currency == "RUR"`, иначе `NaN`.
3. Выведите первые строки со столбцами `salary_from`, `currency`, `salary_rur`, `published_at`.

**Зачем:** в медиану не должны попадать суммы в других валютах. Зарплаты не нужно пересчитывать, заполнять нулями или усреднять между `salary_from` и `salary_to`.

**Методы:** `pd.to_datetime`; для условного столбца подойдёт `np.where` или присваивание через `loc`.

In [ ]:
# 2. Подготовка
# решение

# 1.
vacancies["published_at"] = pd.to_datetime(vacancies["published_at"])

# 2.
vacancies["salary_rur"] = np.where(
    vacancies["currency"] == "RUR",
    vacancies["salary_from"],
    np.nan
)

# 3.
preview_columns = ["salary_from", "currency", "salary_rur", "published_at"]
display(vacancies[preview_columns].head())

,salary_from,currency,salary_rur,published_at
0,35000.0,RUR,35000.0,2026-05-14 06:00:00
1,140000.0,RUR,140000.0,2026-05-29 06:00:00
2,135000.0,RUR,135000.0,2026-06-23 06:00:00
3,40000.0,RUR,40000.0,2026-01-13 06:00:00
4,40000.0,RUR,40000.0,2026-06-03 06:00:00


### 2.1. Сравните заполненность зарплат по валютам

После создания `salary_rur` вернитесь к исходному столбцу `salary_from`. Для каждого значения `currency` выведите:

- число строк;
- число непустых значений `salary_from`;
- среднее значение `salary_from`.

Используйте один `groupby` по валюте, отбор `salary_from` и список агрегатных функций `size`, `count`, `mean`. Включите в результат и строки с неизвестной валютой: для этого в `groupby` есть параметр `dropna=False`.

Коротко поясните разницу между `size` и `count`. Значения разных валют между собой усреднять не нужно; здесь каждая группа считается отдельно.

In [ ]:
# 2.1. Зарплаты по валютам
# решение

currency_stats = (
    vacancies
    .groupby("currency", dropna=False)["salary_from"]
    .agg(["size", "count", "mean"])
)

print("Статистика по валютам (размер, количество непустых, среднее):")
display(currency_stats)

Статистика по валютам (размер, количество непустых, среднее):


,size,count,mean
currency,,,
RUR,44,44,76590.909091
NaN,8,0,NaN


size считает все строки в группе включая пропуски, а count только непустые значения. это важно для расчёта доли заполненных зарплат

## 3. Соберите таблицу вакансий со справочниками

Получите DataFrame `df`, в котором **одна строка по-прежнему соответствует одной вакансии**.

Последовательно добавьте к `vacancies`:

1. `employer_name` из `employers` — по `employer_id`. Из справочника работодателей возьмите только ID и название, чтобы не добавить второй `area_id`.
2. `area_name` из `areas` — по `area_id`. Переименуйте `area_name` в `city`.
3. `size_group` из `size_groups` — по `employer_id`. Пропущенную группу заполните строкой `"unknown"`.

Во всех трёх случаях используйте **левое соединение**, чтобы сохранить все вакансии.

**Что вывести:** размер `df` и первые строки со столбцами `vacancy_id`, `employer_name`, `city`, `size_group`. После объединений должно остаться **52 строки**.

**Методы:** выбор списка столбцов, `merge`, `rename`, `fillna`.

In [ ]:
# 3. Вакансии и справочники
# решение

# 1.
df = vacancies.merge(
    employers[["employer_id", "employer_name"]],
    on="employer_id",
    how="left",
    validate="m:1",
)

# 2.
df = df.merge(
    areas,
    on="area_id",
    how="left",
    validate="m:1",
).rename(columns={"area_name": "city"})

# 3.
df = df.merge(
    size_groups,
    on="employer_id",
    how="left",
    validate="m:1",
)

df["size_group"] = df["size_group"].fillna("unknown")


print(f"Размер df: {df.shape}")
display(df[["vacancy_id", "employer_name", "city", "size_group"]].head())

Размер df: (52, 17)


,vacancy_id,employer_name,city,size_group
0,1001,ТехноБанк,Москва,large
1,1002,ТехноБанк,Москва,large
2,1003,ТехноБанк,Москва,large
3,1004,ТехноБанк,Москва,large
4,1005,Мосмаркет,Москва,large


## 4. Добавьте число откликов и приглашений на вакансию

1. В `responses` создайте `is_invited`: `True`, если `status == "invited"`, иначе `False`.
2. Сгруппируйте отклики по `vacancy_id`. Получите таблицу `response_counts` с двумя показателями:
   - `responses` — количество `response_id`;
   - `invited` — сумма `is_invited` (`True` при суммировании считается как 1).
3. Присоедините эту таблицу к `df` по `vacancy_id` левым соединением.
4. В столбцах `responses` и `invited` заполните пропуски нулями: здесь они означают отсутствие откликов.
5. Выведите первые строки с ID вакансии и двумя новыми показателями. Посчитайте, у скольких вакансий `responses == 0`.

**Важно:** сначала считаем отклики на вакансию, затем соединяем. Прямое присоединение исходных откликов повторит строки вакансий, и их количество в отчёте станет неверным.

**Методы:** сравнение, `groupby`, именованное `agg`, `reset_index`, `merge`, `fillna`, маска или сумма логического условия.

**Самопроверка:** в `df` всё ещё 52 строки, сумма `responses` — **628**, сумма `invited` — **92**, без откликов — **10 вакансий**. Для проверки достаточно вывести числа; писать `assert` не требуется.

In [ ]:
# 4. Отклики на вакансию
# решение

# 1.
responses["is_invited"] = responses["status"] == "invited"

# 2.
response_counts = (
    responses
    .groupby("vacancy_id", as_index=False)
    .agg(
        responses=("response_id", "count"),
        invited=("is_invited", "sum"),
    )
)

display(response_counts.head())

# 3.
df = df.merge(response_counts, on="vacancy_id", how="left", validate="m:1")

# 4.
df["responses"] = df["responses"].fillna(0).astype(int)
df["invited"] = df["invited"].fillna(0).astype(int)

# 5.
display(df[["vacancy_id", "responses", "invited"]].head())

print("Число строк в df:", len(df))
print("Сумма responses:", df["responses"].sum())
print("Сумма invited:", df["invited"].sum())
print("Вакансий без откликов (responses == 0):", (df["responses"] == 0).sum())

,vacancy_id,responses,invited
0,1001,25,3
1,1002,19,1
2,1003,25,0
3,1004,46,6
4,1005,32,7


,vacancy_id,responses,invited
0,1001,25,3
1,1002,19,1
2,1003,25,0
3,1004,46,6
4,1005,32,7


Число строк в df: 52
Сумма responses: 628
Сумма invited: 92
Вакансий без откликов (responses == 0): 10


## 5. Постройте отчёт по городам

Создайте `city_report`, сгруппировав `df` по `city`.

| Столбец отчёта | Как посчитать |
|---|---|
| `city` | Город вакансии |
| `vacancies` | Количество `vacancy_id` |
| `responses` | Сумма `responses` |
| `invited` | Сумма `invited` |
| `median_salary` | Медиана `salary_rur` |
| `salary_count` | Число непустых значений `salary_rur` |
| `salary_share` | `salary_count / vacancies` в готовом отчёте |
| `invitation_share` | `invited / responses` в готовом отчёте |

Отсортируйте отчёт по числу откликов по убыванию и выведите целиком.

**Подсказка:** в одном `groupby().agg()` можно задать название каждого нового столбца и пару «исходный столбец, функция». После агрегации используйте `reset_index()` или заранее задайте `as_index=False`.

Количество и доля непустых зарплат относятся к `salary_rur`: это известные нижние суммы в рублях. Знаменатель `salary_share` — **все вакансии группы**, включая вакансии без зарплаты.

Долю приглашений считайте **из итоговых чисел по городу**, а не как среднее долей отдельных вакансий. В случае 0 приглашений и 0 откликов обычное деление даёт `NaN`; нулём его заменять не нужно.

**Самопроверка:** 5 городов; сумма `vacancies` — 52, сумма `responses` — 628.

In [ ]:
# 5. Отчёт по городам
# решение

# 1.
city_report = df.groupby("city", as_index=False).agg(
    vacancies=("vacancy_id", "count"),
    responses=("responses", "sum"),
    invited=("invited", "sum"),
    median_salary=("salary_rur", "median"),
    salary_count=("salary_rur", "count")
)

# 2.
city_report["salary_share"] = city_report["salary_count"] / city_report["vacancies"]
city_report["invitation_share"] = city_report["invited"] / city_report["responses"]

# 3.
city_report = city_report.sort_values("responses", ascending=False).reset_index(drop=True)

# 4.
display(city_report)

print(f"Количество городов: {city_report['city'].nunique()} (ждем 5)")
print(f"Сумма vacancies: {city_report['vacancies'].sum()} (ждем 52)")
print(f"Сумма responses: {city_report['responses'].sum()} (ждем 628)")

,city,vacancies,responses,invited,median_salary,salary_count,salary_share,invitation_share
0,Москва,26,405,54,52500.0,22,0.846154,0.133333
1,Екатеринбург,7,80,16,60000.0,6,0.857143,0.200000
2,Санкт-Петербург,9,62,10,60000.0,9,1.000000,0.161290
3,Казань,4,54,6,80000.0,3,0.750000,0.111111
4,Новосибирск,6,27,6,50000.0,4,0.666667,0.222222


Количество городов: 5 (ждем 5)
Сумма vacancies: 52 (ждем 52)
Сумма responses: 628 (ждем 628)


## 6. Постройте такой же отчёт по размеру работодателя

Создайте `size_report` с теми же семью показателями, что в предыдущем пункте, но группируйте по `size_group`.

1. Сохраните группу `unknown`: неизвестный размер компании не означает, что её вакансии и отклики нужно исключить.
2. Отсортируйте отчёт по `invitation_share` по убыванию и выведите его.
3. Посмотрите, какая из известных групп (`small`, `medium`, `large`) имеет наибольшую долю приглашений.

**Подсказка:** повторите логику предыдущего пункта с другой колонкой группировки. Выносить её в отдельную функцию не нужно.

**Самопроверка:** 4 группы, включая `unknown`; итоговые количества вакансий, откликов и приглашений совпадают с отчётом по городам.

In [ ]:
# 6. Отчёт по размеру работодателя
#  решение

# 1.
size_report = df.groupby("size_group", as_index=False).agg(
    vacancies=("vacancy_id", "count"),
    responses=("responses", "sum"),
    invited=("invited", "sum"),
    median_salary=("salary_rur", "median"),
    salary_count=("salary_rur", "count")
)

# 2.
size_report["salary_share"] = size_report["salary_count"] / size_report["vacancies"]
size_report["invitation_share"] = size_report["invited"] / size_report["responses"]

# 3.
size_report = size_report.sort_values("invitation_share", ascending=False).reset_index(drop=True)

# 4.
display(size_report)

print(f"Количество групп: {size_report['size_group'].nunique()} (ждем 4, включая unknown)")
print(f"Сумма vacancies: {size_report['vacancies'].sum()} (ждем 52)")
print(f"Сумма responses: {size_report['responses'].sum()} (ждем 628)")
print(f"Сумма invited: {size_report['invited'].sum()} (ждем 92)")

known_groups = size_report[size_report["size_group"] != "unknown"]
best_known_group = known_groups.loc[known_groups["invitation_share"].idxmax(), "size_group"]
max_invite_share = known_groups["invitation_share"].max()

print(f"Среди известных групп наибольшую долю приглашений имеет '{best_known_group}' ({max_invite_share:.1%}).")

,size_group,vacancies,responses,invited,median_salary,salary_count,salary_share,invitation_share
0,medium,13,134,25,55000.0,12,0.923077,0.186567
1,large,25,287,43,50000.0,22,0.880000,0.149826
2,unknown,2,24,3,135000.0,1,0.500000,0.125000
3,small,12,183,21,75000.0,9,0.750000,0.114754


Количество групп: 4 (ждем 4, включая unknown)
Сумма vacancies: 52 (ждем 52)
Сумма responses: 628 (ждем 628)
Сумма invited: 92 (ждем 92)
Среди известных групп наибольшую долю приглашений имеет 'medium' (18.7%).


## 7. Оформите таблицу, постройте график и сохраните отчёт

1. Отобразите `city_report` через `.style.format()`: обе доли (`invitation_share` и `salary_share`) — в процентах с одним знаком после запятой, медианная зарплата — без дробной части, пропуски — прочерком. Остальные украшения необязательны.
2. Постройте столбчатый график Plotly Express по `city_report`: по X — город, по Y — количество откликов. Добавьте понятный заголовок.
3. Сохраните `city_report` в `outputs/vacancies_report_v2.xlsx` без индекса DataFrame.

**Методы:** `style.format` с форматами `"{:.1%}"` и `"{:,.0f}"`, параметр `na_rep`; `px.bar`; `to_excel`.

В Excel достаточно сохранить обычную таблицу. Переносить туда оформление Styler не требуется.

In [ ]:
# 7. Оформление таблицы
# решение
import plotly.express as px
from pathlib import Path

# Создаём папку outputs, если её нет
output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)


styled_city_report = city_report.style.format(
    {
        "invitation_share": "{:.1%}",
        "salary_share": "{:.1%}",
        "median_salary": "{:,.0f}",
    },
    na_rep="—",
)

display(styled_city_report)

,city,vacancies,responses,invited,median_salary,salary_count,salary_share,invitation_share
0,Москва,26,405,54,"52,500",22,84.6%,13.3%
1,Екатеринбург,7,80,16,"60,000",6,85.7%,20.0%
2,Санкт-Петербург,9,62,10,"60,000",9,100.0%,16.1%
3,Казань,4,54,6,"80,000",3,75.0%,11.1%
4,Новосибирск,6,27,6,"50,000",4,66.7%,22.2%


In [ ]:
# 7. График и Excel
#решение

fig = px.bar(
    city_report,
    x="city",
    y="responses",
    color="invited",
    text="responses",
    title="Количество откликов на вакансии по городам",
    labels={
        "city": "Город",
        "responses": "Количество откликов",
        "invited": "Приглашений",
    },
)

fig.update_traces(textposition="outside")
fig.update_layout(
    xaxis_title="Город",
    yaxis_title="Количество откликов",
    showlegend=True,
)

fig.show()


excel_path = output_dir / "vacancies_report_v2.xlsx"
city_report.to_excel(excel_path, index=False)

print(f"\nотчет сохранён: {excel_path}")


отчет сохранён: outputs/vacancies_report_v2.xlsx


## 8. Ответьте на три вопроса

Достаточно 1–2 предложений на каждый вопрос.

1. В каком городе больше всего откликов? Сколько там откликов и вакансий?
2. У какой известной группы размера работодателя самая высокая доля приглашений? Укажите долю и числа приглашений и откликов, по которым она рассчитана.
3. Почему полученную медиану нельзя назвать «медианной зарплатой всех вакансий»? Укажите, какие значения использованы и какие не вошли в расчёт.

**Готовность основной части:** есть таблица на уровне вакансии, два отчёта, форматированная таблица, график, Excel-файл и три ответа. Контрольные количества совпадают.

**ответы:**

1. Больше всего откликов в Москве: там зафиксировано 405 откликов на 26 вакансий.

2. Самая высокая доля приглашений среди известных групп у работодателей размера `medium` (18,7%). Эта доля рассчитана как отношение 25 приглашений к 134 общим откликам в этой группе.

3. Рассчитанная медиана отражает только известные нижние границы зарплат (`salary_from`), выраженные строго в рублях (`RUR`). В расчёт не вошли вакансии с пропущенными значениями зарплаты, вакансии, где указана только верхняя граница (`salary_to`), а также вакансии в других валютах (если бы они были).


## Дополнительно, по желанию

### А. Сводная «город × размер работодателя»

По `df` постройте широкую таблицу: строки — `city`, столбцы — `size_group`, значения — количество `vacancy_id`. Отсутствующие сочетания заполните нулями.

**Метод:** `pivot_table`, агрегатная функция `count`.

### Б. Второй график

Постройте столбчатую диаграмму доли приглашений по `size_report`, исключив `unknown`. Добавьте в подсказку при наведении `invited` и `responses`, чтобы вместе с долей видеть объём данных.

**Методы:** фильтр по условию, `px.bar`, `hover_data`.

Эти два упражнения не нужны для завершения основной части.

In [ ]:
# Дополнительно А. Сводная таблица
# решение

pivot_city_size = df.pivot_table(
    index="city",
    columns="size_group",
    values="vacancy_id",
    aggfunc="count",
    fill_value=0
)

pivot_city_size.columns.name = None

print("Сводная таблица: Вакансии по городам и размеру работодателя")
display(pivot_city_size)

Сводная таблица: Вакансии по городам и размеру работодателя


,large,medium,small,unknown
city,,,,
Екатеринбург,0,7,0,0
Казань,1,0,3,0
Москва,17,0,7,2
Новосибирск,4,0,2,0
Санкт-Петербург,3,6,0,0


In [ ]:
# Дополнительно Б. Доли приглашений на графике
# решение

# 1. Фильтруем size_report, исключая группу "unknown"
known_sizes = size_report[size_report["size_group"] != "unknown"].copy()

# 2. Строим столбчатую диаграмму
fig_size = px.bar(
    known_sizes,
    x="size_group",
    y="invitation_share",
    text="invitation_share",
    hover_data=["invited", "responses"],
    title="Доля приглашений по размеру работодателя (без unknown)",
    labels={
        "size_group": "Размер работодателя",
        "invitation_share": "Доля приглашений",
        "invited": "Приглашений",
        "responses": "Откликов",
    },
    color="size_group",
    color_discrete_map={
        "small": "#636EFA",
        "medium": "#EF553B",
        "large": "#00CC96",
    },
)

# 3. Форматируем подписи на столбцах как проценты
fig_size.update_traces(
    texttemplate="%{text:.1%}",
    textposition="outside",
    hovertemplate=(
        "<b>%{x}</b><br>"
        "Доля приглашений: %{y:.1%}<br>"
        "Приглашений: %{customdata[0]}<br>"
        "Откликов: %{customdata[1]}<extra></extra>"
    ),
)

# 4. Настраиваем оси и легенду
fig_size.update_layout(
    yaxis_tickformat=".0%",
    xaxis_title="Размер работодателя",
    yaxis_title="Доля приглашений",
    showlegend=False,
    margin_t=60,
)

fig_size.show()